In [6]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, List, Dict, Any, Literal
import json, os

In [ ]:
STRUCTURED_TEMPLATE = {
  "active_problems": [
    {
      "problem": "",
      "status": "",
      "evidence": [
        {
          "source_type": "note | discharge | imaging",
          "date": "",
          "snippet": ""
        }
      ]
    }
  ],
  "recent_clinical_course": [
    {
      "event": "",
      "date_range": "",
      "evidence": []
    }
  ],
  "medications": {
    "current": [],
    "high_risk": []
  },
  "allergies": [],
  "key_results": [],
  "imaging_summary": [
    {
      "study": "",
      "date": "",
      "impression": ""
    }
  ],
  "procedures": [],
  "pending_items": []
}

class AgentState(TypedDict):
    template: Dict[str, Any]
    action: Literal["search", "finish"]
    next_query: str | None
    
    retrieved_docs_old: List[str]
    retrieved_docs_new: List[str]
    action_history: List[Dict[str, Any]]
    step: int


MAX_STEPS = 10
initial_state = {
    "template": STRUCTURED_TEMPLATE,
    "action": None,
    "next_query": None,
    "retrieved_docs_old": [],
    "retrieved_docs_new": [],
    "action_history": [],
    "step": 0
}

def reason_and_plan(state: AgentState) -> Dict:
    if state["step"] == MAX_STEPS:
        return {"action": "finish", "next_query": None, "action_history": state["action_history"] + ["finish"]}

    SYSTEM_PROMPT = """\
    SYSTEM INSTRUCTION: think silently if needed.
    You are an autonomous clinical extraction agent.
    Your task is to provide queries to the vectorstore containing
    patient's medical history. The retrieved documents will be used
    to fill in the template provided by the user.
    
    You ALWAYS:
    1. Inspect the template
    2. Review past actions
    3. Decide the next action
    4. Return STRICT JSON
    
    Return STRICT JSON:
    {
      "thought": "<brief reasoning>",
      "action": "search" | "finish",
      "query": "<search query or None>"
    }
    """
    
    USER_PROMPT = f"""\
    Template:
    {state['template']}
    
    Past actions:
    {state['action_history']}
    """
    
    messages = [
        {
            "role": "system",
            "content": [
                {"type": "text", "text": SYSTEM_PROMPT}
            ],
        },
        {
            "role": "user",
            "content": [
                {"type": "text", "text": USER_PROMPT}
            ],
        },
    ]

    response = pipe(messages, max_new_tokens=5000)
    print(response[0]["generated_text"])
    clean = re.sub(r"^```json\s*|\s*```$", "", response[0]["generated_text"][-1]["content"].split("<unused95>")[1].strip())
    plan = json.loads(clean)

    if plan["action"] == "finish":
        action_history = state["action_history"] + ["finish"]
    else:
        action_history = state["action_history"] + [plan['action'] + " - Query: " + plan['query']]
    
    return {
        "action": plan["action"],
        "next_query": plan["query"],
        "action_history": action_history,
        "step": state["step"] + 1
    }

def vector_search(state: AgentState) -> Dict:
    results = text_vectorstore.similarity_search(state["next_query"], k=3)
    snippets = [r.page_content for r in results]

    return {
        "retrieved_docs_new": snippets
    }

def update_template(state: AgentState) -> Dict:
    SYSTEM_PROMPT = """\
    SYSTEM INSTRUCTION: think silently if needed.
    You are an autonomous clinical extraction agent.
    Your task is to fill in the template provided by
    the user with the newely retrieved documents
    
    You ALWAYS:
    1. Inspect the template
    2. Review the retrieved documents
    3. Updae the template based on new documents
    4. Return the template in STRICT JSON
    """
    
    USER_PROMPT = f"""\

    New Retrieved documents:
    {state['retrieved_docs_new']}

    Template:
    {state['template']}
    """
    
    messages = [
        {
            "role": "system",
            "content": [
                {"type": "text", "text": SYSTEM_PROMPT}
            ],
        },
        {
            "role": "user",
            "content": [
                {"type": "text", "text": USER_PROMPT}
            ],
        },
    ]

    response = pipe(messages, max_new_tokens=5000)
    print(response[0]["generated_text"])
    clean = re.sub(r"^```json\s*|\s*```$", "", response[0]["generated_text"][-1]["content"].split("<unused95>")[1].strip())
    template = json.loads(clean)

    return {"template": template, "retrieved_docs_old": state["retrieved_docs_old"] + state["retrieved_docs_new"], "retrieved_docs_new": []}

def route(state: AgentState):
    return state["action"]

graph = StateGraph(AgentState)

graph.add_node("reason_and_plan", reason_and_plan)
graph.add_node("vector_search", vector_search)
graph.add_node("update_template", update_template)

graph.add_edge(START, "reason_and_plan")

graph.add_conditional_edges(
    "reason_and_plan",
    route,
    {
        "search": "vector_search",
        "finish": END
    }
)

graph.add_edge("vector_search", "update_template")
graph.add_edge("update_template", "reason_and_plan")

compiled = graph.compile()

result = compiled.invoke(initial_state)
result